Packages Download

In [19]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio pandas

Resolved 75 packages in 249ms                                        
Checked 75 packages in 1ms


In [20]:
from dotenv import load_dotenv
import os

from openai import AsyncAzureOpenAI
from agents import (
    Agent,
    Runner,
    function_tool,
    OpenAIChatCompletionsModel,
    set_tracing_disabled,
    SQLiteSession,
    handoff,
)
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
from pypdf import PdfReader
import gradio as gr
import pandas as pd

API Key Checkover.

In [21]:
load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

Endpoint loaded: True
API key loaded: True


Disabling Tracer.

In [22]:
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)

print("Azure model configured for the OpenAI Agents SDK.")

Azure model configured for the OpenAI Agents SDK.


Monthly Expenses collection Tool.

In [ ]:
def extract_pdf_text(file_path: str) -> str:
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text


MONTHLY_EXPENSES_COLLECTION = {
    "monthly_expenses_pdf_1": {
        "filename": "MonthlySpendingData_Placeholder.pdf",
        "content": extract_pdf_text("data/MonthlySpendingData_Placeholder.pdf")
    }
}


spending_data_text = MONTHLY_EXPENSES_COLLECTION["monthly_expenses_pdf_1"]["content"]
print("Loaded PDF text length:", len(spending_data_text), "characters")

Loaded PDF text length: 275 characters


Here, our agent will have a csv parsing tool.
Paint points: different banks have different csv format. The sample csv is a placeholder

In [24]:
def parse_csv_transactions_tool(file_path: str) -> dict:
    """Parse and validate transactions from a CSV file."""
    try:
        df = pd.read_csv(file_path)

        required_columns = ["date", "description", "amount"]
        missing = [col for col in required_columns if col not in df.columns]

        if missing:
            return {"success": False, "error": f"Missing columns: {missing}"}

        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

        invalid_rows = df[["date", "amount"]].isna().any(axis=1)
        if invalid_rows.any():
            return {
                "success": False,
                "error": f"{invalid_rows.sum()} rows contain invalid dates or amounts"
            }

        return {
            "success": True,
            "transaction_count": len(df),
            "transactions": df.assign(
                date=df["date"].dt.strftime("%Y-%m-%d")
            ).to_dict(orient="records")
        }

    except (OSError, pd.errors.ParserError, UnicodeError) as e:
        return {"success": False, "error": str(e)}

In [25]:
#calculate_spending_summary_tool()

In [26]:
#save_transactions_tool()

Tool Prompt.

In [27]:
instructions = f"""{RECOMMENDED_PROMPT_PREFIX}

You are Expense Analyst Agent, You are a Financial Advizing Agent that Analyzes financial data provided by users through 
CSV files, chat inputs, PDF files. You categorizes transactions, identifies spending patterns, and summarizes expenses for the user


You are to, when prompted by the user, to list the monthly expenses provided in the pdf from the MONTHLY_EXPENSES_COLLECTION and tell them how they could better optimize their spending.

. 

"""

ExpenseAnalystAgent = Agent(
    name="ExpenseAnalystAgent",
    handoff_description="Analyzes budget and summarizes spending.",
    instructions=instructions,
    model=azure_model,
)

print("Agent built:", ExpenseAnalystAgent.name)

#when executed with tools that can read the MONTHLY_EXPENSES_COLLECTION, the chatbot should be able to properly assess the file
#for now, its a pdf as a placeholder but we can change that later. 
# We will add more to the description of the agent we are creating the tools - Chris


Agent built: ExpenseAnalystAgent


CHAT WITH GRADIO!

In [28]:
chat_session = SQLiteSession("gradio_user", "user_expenses.db")

async def chat(message, history):
    result = await Runner.run(ExpenseAnalystAgent, message, session=chat_session)
    return result.final_output

gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [29]:
# Hey guys, so this is basically the skeleton of the Expenses Agent, I formatted it in a way that it will read a single pdf file when given the tools to do so
#We will end up adding memory as well, preferably after the tools are implemented for this agent
#It may be a little thrown together in terms of the structure, but I made what I could on my own. If you guys have issues with it, we will modify it together. 
#-Chris